# Review proposed claim interpretations

Read docs/SEMANTIC_CLAIM_ADDENDUM_DRAFT_v1.md first. Proposals are AI-assisted development notes, not confirmed labels. All 21 documents were discussed; 27 selected propositions are illustrative, not exhaustive. The original worksheet, source corpus and frozen annotations remain unchanged.


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/processed/model_corpus_v1.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project root or notebooks directory.")
source = ROOT / "data/processed/model_corpus_v1.csv"
if hashlib.sha256(source.read_bytes()).hexdigest() != "d66a330b121ae5e59d96b58fb12d34b0c4fdbd9059d427abdc7bd182b07bcc2c":
    raise ValueError("Frozen corpus hash mismatch.")
corpus = pd.read_csv(source, keep_default_na=False).set_index("execution_number")
proposals = pd.read_csv(ROOT / "reports/semantic_development_proposals_v1.csv", keep_default_na=False)
coverage = pd.read_csv(ROOT / "reports/semantic_development_document_coverage_v1.csv", keep_default_na=False)
selection = pd.read_csv(ROOT / "data/processed/semantic_development_sample_v1.csv", keep_default_na=False)
if coverage.execution_number.tolist() != selection.execution_number.tolist():
    raise ValueError("Fixed sample IDs or order differ; run notebook 07 first.")
if not proposals.claim_id.is_unique or len(coverage) != 21:
    raise ValueError("Unexpected proposal IDs or document coverage.")
for row in proposals.itertuples():
    text = corpus.loc[row.execution_number, "model_text"]
    if (hashlib.sha256(text.encode()).hexdigest() != row.context_sha256
            or text[row.span_start:row.span_end] != row.evidence_span
            or text.count(row.evidence_span) != 1):
        raise ValueError(f"Evidence mismatch: {row.claim_id}")
print(f"Verified {len(proposals)} exact spans; 21 documents covered; all decisions pending.")

path = ROOT / "data/processed/semantic_development_adjudication_draft_v1.csv"
owner_columns = ["owner_decision", "owner_claim_source", "owner_event_type",
                 "owner_event_actor", "owner_polarity", "owner_statement_mode", "owner_note"]
if path.exists():
    worksheet = pd.read_csv(path, keep_default_na=False)
    if set(worksheet.columns) != set(proposals.columns) | set(owner_columns):
        raise ValueError("Existing worksheet schema differs; nothing overwritten.")
    pd.testing.assert_frame_equal(worksheet[proposals.columns], proposals, check_dtype=False)
    print("Existing owner worksheet retained; no decisions overwritten.")
else:
    worksheet = proposals.copy()
    for column in owner_columns:
        worksheet[column] = ""
    with path.open("x", encoding="utf-8", newline="") as handle:
        worksheet.to_csv(handle, index=False)
    print("Created separate owner worksheet with blank decision fields.")
display(coverage)


## Inspect and decide

Use show_claim_review(231), then other IDs. In the separate CSV, owner_decision may be accept, revise or defer. Acceptance means agreement with the proposed selected-proposition interpretation, not factual verification. Fill owner fields only for revisions and explain them in owner_note. Do not infer accuracy or agreement rates from this discussion. Save the CSV in your spreadsheet editor; reruns retain it. Document-level exclusions also need owner review before a final reference dataset is produced.


In [ ]:
def show_claim_review(execution_number):
    if execution_number not in set(coverage.execution_number):
        raise ValueError("Execution is outside this fixed sample.")
    print(corpus.loc[execution_number, "model_text"])
    columns = ["claim_id", "evidence_span", "proposed_claim_source", "proposed_event_type",
               "proposed_event_actor", "proposed_polarity", "statement_mode", "event_reference"]
    display(proposals.loc[proposals.execution_number.eq(execution_number), columns])
    if not proposals.execution_number.eq(execution_number).any():
        print(coverage.set_index("execution_number").loc[execution_number, "selection_note"])

show_claim_review(231)
